# grad-expressed-in-out — ex1: write sigmoid_back using cached out (no second sigmoid call)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-expressed-in-out`. Running the final beacon cell reports progress against the `Backprop: grad expressed in out` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grad expressed in out` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-expressed-in-out`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-expressed-in-out"
DD_SUBTOPIC = "Backprop: grad expressed in out"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Grad expressed in `out` — quick refresher

Many elementwise back fns can be written in terms of the CACHED forward `out` instead of recomputing the activation:

```
sigmoid_back(grad_out, out, x) = grad_out * out * (1 - out)
tanh_back   (grad_out, out, x) = grad_out * (1 - out**2)
exp_back    (grad_out, out, x) = grad_out * out
```

The point of the `(grad_out, out, x)` signature is exactly to make this possible — every back fn receives the cached forward output `out`, so it never has to call `sigmoid(x)` or `exp(x)` a second time on the reverse pass. The savings compound: one allocation + one exp per node, multiplied by every elementwise op in the graph.

Numerical stability bonus: `out * (1 - out)` and `1 - out**2` are both bounded in `[0, 0.25]` and `[0, 1]` respectively, with no intermediate exponentials — far better behaved than `exp(-x) / (1 + exp(-x))**2`.

### Exercise 1 — write sigmoid_back using cached out (no second sigmoid call)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the 'grad expressed in out' pattern by writing sigmoid_back as grad_out * out * (1 - out) — reusing the cached forward output rather than recomputing sigmoid(x).
> Keywords: sigmoid, cached-out, elementwise, no-recompute
> ```

**KCs targeted:** `grad-expressed-in-out`, `back-fn-uses-cached-out`

Implement `sigmoid_back(grad_out, out, x)` — the exemplar of the 'grad expressed in `out`' pattern.

**Math.** `out = sigmoid(x) = 1 / (1 + exp(-x))`. The derivative factors cleanly through the output:

```
d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x))
                = out * (1 - out)
```

So by the chain rule:

```
dL/dx = grad_out * out * (1 - out)
```

**The point of this drill.** You **must use `out`**, not `t.sigmoid(x)`. The whole reason the back-fn signature passes `out` is so we never recompute the activation on the reverse pass. The test inspects the function body to make sure you didn't sneak in a second `sigmoid` call.

**Inputs.** Plain `torch.Tensor`, same shape; no autograd. Float dtype. Output: tensor with the same shape as `x`.

**Tip.** One line is enough. The clarity of the cached-`out` form is the lesson — `out * (1 - out)` reads exactly like the math.

In [ ]:
def sigmoid_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    """Gradient of out = sigmoid(x), expressed via the cached `out`."""
    raise NotImplementedError()


def _test_ex1():
    # --- numerical correctness ---
    x = t.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
    out = t.sigmoid(x)
    grad_out = t.ones(5)
    g = sigmoid_back(grad_out, out, x)
    expected = out * (1 - out)
    assert g.shape == x.shape, f'shape: {g.shape}'
    assert t.allclose(g, expected), f'value: {g} vs {expected}'

    # --- non-unit grad_out scales each entry by the chain rule ---
    grad_out = t.tensor([5.0, -3.0, 2.0, 0.5, -1.0])
    g = sigmoid_back(grad_out, out, x)
    expected = grad_out * out * (1 - out)
    assert t.allclose(g, expected), 'chain-rule scaling failed'

    # --- matrix shape ---
    rng = t.Generator().manual_seed(0)
    X = t.randn(3, 4, generator=rng)
    G = t.randn(3, 4, generator=rng)
    out_mat = t.sigmoid(X)
    g_mat = sigmoid_back(G, out_mat, X)
    assert g_mat.shape == (3, 4)
    assert t.allclose(g_mat, G * out_mat * (1 - out_mat))

    # --- witness vs torch.autograd ---
    x_ref = t.tensor([-1.5, -0.2, 0.3, 1.5], requires_grad=True)
    y = t.sigmoid(x_ref).sum()
    y.backward()
    out_cached = t.sigmoid(x_ref.detach())
    g_ours = sigmoid_back(t.ones(4), out_cached, x_ref.detach())
    assert t.allclose(g_ours, x_ref.grad, atol=1e-6), (
        f'sigmoid_back disagrees with autograd: ours={g_ours}, ref={x_ref.grad}'
    )

    # --- THE point of this atom: must use `out`, NOT recompute sigmoid(x) ---
    # Behavioural witness: pass a DELIBERATELY WRONG `out` (not equal to sigmoid(x))
    # and check the function trusts `out` rather than recomputing from `x`.
    # A correct implementation returns grad_out * fake_out * (1 - fake_out);
    # a recompute-from-x implementation would ignore fake_out and return the
    # true sigmoid derivative — easy to distinguish.
    fake_x = t.tensor([0.0, 0.0, 0.0])
    fake_out = t.tensor([0.25, 0.5, 0.75])   # NOT what sigmoid(0) is (= 0.5)
    got = sigmoid_back(t.ones(3), fake_out, fake_x)
    expected_from_fake_out = fake_out * (1 - fake_out)
    assert t.allclose(got, expected_from_fake_out), (
        'sigmoid_back must use the cached `out`, not recompute sigmoid(x). '
        f'Given fake_out={fake_out.tolist()} the result should be '
        f'{expected_from_fake_out.tolist()}; got {got.tolist()}.'
    )

    # --- robustness: works on a scalar too ---
    x_sc = t.tensor(0.0)
    out_sc = t.sigmoid(x_sc)
    g_sc = sigmoid_back(t.tensor(1.0), out_sc, x_sc)
    assert g_sc.shape == x_sc.shape
    # At x=0, sigmoid=0.5, derivative = 0.5 * 0.5 = 0.25.
    assert abs(g_sc.item() - 0.25) < 1e-6, f'scalar case: {g_sc}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def sigmoid_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # The cached `out` is sigmoid(x); we reuse it instead of recomputing.
    # d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x)) = out * (1 - out).
    return grad_out * out * (1 - out)
```

**Why `out`, not `x`, drives the formula.** The local derivative of sigmoid happens to factor as `out * (1 - out)`. Other activations that share this property: `tanh_back` uses `1 - out**2`, `exp_back` uses `out` directly. The shared `(grad_out, out, x)` signature is engineered so any of these can write the cleanest expression.

**Cost savings.** A second `t.sigmoid(x)` call would allocate a new tensor and run another exp+division per element. For a deep network with millions of activations, that's a measurable hit on the backward pass.

**Why pass `x` at all then?** Because not every op can be expressed in terms of `out`. `relu_back` needs `x > 0` (the cached `out = max(x, 0)` doesn't tell you whether `x` was positive at 0). Keeping `x` in the signature is the uniform-dispatch tax.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()